In [ ]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!apt-get install -y -qq openmpi-bin libopenmpi-dev > /dev/null
!pip install -q mpi4py numba numpy

In [ ]:
GITHUB_USERNAME = "Koreshzy0"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME}")

In [ ]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import sys
import json
import math
import re
import time
import hashlib
import platform
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, njit, float64, uint64, types
from numba.extending import intrinsic
from numba.core import cgutils
from llvmlite import ir

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2
GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_N = 256
PAD_N = LOCAL_N + 2
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
LAMBDA_DAMP = 0.15
CFL_SAFETY = 0.8
SHM_SLOTS = 64
SHM_REPS = 3


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def detect_cpu_line_size() -> int:
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


# ==============================================================================
# TASK 1: Hybrid MPI Cartesian Grid + Node-Local MPI-3 Shared-Memory Window
# ==============================================================================
def init_hybrid_context(github_user: str) -> dict:
    comm = MPI.COMM_WORLD
    world_rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")

    cuda.select_device(0)
    gpu_uuid = comm.bcast(get_gpu_uuid() if world_rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}"
    line_bytes = detect_cpu_line_size()

    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}".encode("utf-8")
    ).digest()
    instance_seed = int.from_bytes(digest[:8], "big")

    # --- non-periodic 2x2 Cartesian communicator
    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    rank = cart_comm.Get_rank()
    coords = cart_comm.Get_coords(rank)
    up, down = cart_comm.Shift(0, 1)      # (source, dest) along rows
    left, right = cart_comm.Shift(1, 1)   # (source, dest) along columns

    # --- node-local shared-memory communicator + 64-byte aligned window
    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    extra = 8  # 8 spare uint64 slots so we can slide to a 64-byte boundary
    nbytes = (SHM_SLOTS + extra) * 8 if node_rank == 0 else 0
    win = MPI.Win.Allocate_shared(nbytes, 8, comm=node_comm)
    buf, _ = win.Shared_query(0)
    full = np.ndarray(buffer=buf, dtype=np.uint64, shape=(SHM_SLOTS + extra,))
    addr = full.__array_interface__["data"][0]
    off = node_comm.bcast(((-addr) % 64) // 8 if node_rank == 0 else None, root=0)
    shm_view = full[off:off + SHM_SLOTS]
    if shm_view.__array_interface__["data"][0] % 64 != 0:
        raise RuntimeError("Shared window slice is not 64-byte aligned on this rank.")
    win.Lock_all(MPI.MODE_NOCHECK)
    if node_rank == 0:
        shm_view[:] = 0
    win.Sync()
    node_comm.Barrier()
    win.Sync()

    return dict(
        comm=comm, cart_comm=cart_comm, rank=rank, coords=coords,
        up=up, down=down, left=left, right=right,
        node_comm=node_comm, node_rank=node_rank, win=win, shm_view=shm_view,
        github_user=github_user.strip(), gpu_uuid=gpu_uuid, cpu_sig=cpu_sig,
        cpu_line_bytes=line_bytes, seed=instance_seed,
    )


# ==============================================================================
# TASK 2: rank-local fields + 1D halo pack/unpack kernels
# ==============================================================================
def initialize_fields(ctx: dict) -> dict:
    cart = ctx["cart_comm"]
    py, px = ctx["coords"]
    dx = 1.0 / (GLOBAL_NX + 1)
    idx = np.arange(PAD_N)
    y = (py * LOCAL_N + (idx - 1) + 1) * dx      # rows  -> global y
    x = (px * LOCAL_N + (idx - 1) + 1) * dx      # cols  -> global x
    Y, X = np.meshgrid(y, x, indexing="ij")      # global coordinates, no global grid

    rng = np.random.default_rng(ctx["seed"] % (2**63))
    ph = rng.uniform(0.0, 2.0 * math.pi, size=2)

    kappa = 1.0 + 2.0 * (0.5 + 0.5 * np.sin(6 * math.pi * X + ph[0]) * np.sin(4 * math.pi * Y + ph[1]))
    u0 = np.exp(-((X - 0.45) ** 2 + (Y - 0.55) ** 2) / (2 * 0.06 ** 2))
    S = 0.02 * np.exp(-((X - 0.70) ** 2 + (Y - 0.30) ** 2) / (2 * 0.03 ** 2))

    # Dirichlet u = 0 on the physical domain boundary
    if ctx["up"] == MPI.PROC_NULL:
        u0[0, :] = 0.0
    if ctx["down"] == MPI.PROC_NULL:
        u0[-1, :] = 0.0
    if ctx["left"] == MPI.PROC_NULL:
        u0[:, 0] = 0.0
    if ctx["right"] == MPI.PROC_NULL:
        u0[:, -1] = 0.0

    kappa_max = cart.allreduce(float(kappa[1:-1, 1:-1].max()), op=MPI.MAX)
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max
    return dict(
        u0=np.ascontiguousarray(u0), kappa=np.ascontiguousarray(kappa),
        S=np.ascontiguousarray(S), dx=dx, inv_dx2=1.0 / (dx * dx),
        kappa_max=kappa_max, dt_max=dt_max, dt=dt,
    )


@cuda.jit
def pack_halos_kernel(u, sendbuf):
    # sendbuf layout: [0:256] row 256 -> down | [256:512] row 1 -> up
    #                 [512:768] col 256 -> right | [768:1024] col 1 -> left
    k = cuda.grid(1)
    if k < LOCAL_N:
        sendbuf[k] = u[LOCAL_N, k + 1]
        sendbuf[LOCAL_N + k] = u[1, k + 1]
        sendbuf[2 * LOCAL_N + k] = u[k + 1, LOCAL_N]      # strided column
        sendbuf[3 * LOCAL_N + k] = u[k + 1, 1]            # strided column


@cuda.jit
def unpack_halos_kernel(u, recvbuf):
    # recvbuf layout: [0:256] ghost row 0 (from up) | [256:512] ghost row 257 (from down)
    #                 [512:768] ghost col 0 (from left) | [768:1024] ghost col 257 (from right)
    k = cuda.grid(1)
    if k < LOCAL_N:
        u[0, k + 1] = recvbuf[k]
        u[PAD_N - 1, k + 1] = recvbuf[LOCAL_N + k]
        u[k + 1, 0] = recvbuf[2 * LOCAL_N + k]
        u[k + 1, PAD_N - 1] = recvbuf[3 * LOCAL_N + k]


# ==============================================================================
# TASK 3: shared-memory tiled stencil + serialized / dual-stream steps
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, kappa, S, out, dt, inv_dx2, damp, i0, j0, ni, nj):
    tile = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = i0 + cuda.blockIdx.y * TILE_DIM + ty
    j = j0 + cuda.blockIdx.x * TILE_DIM + tx
    active = (i < i0 + ni) and (j < j0 + nj)

    if active:
        tile[ty + 1, tx + 1] = u[i, j]
        if ty == 0:
            tile[0, tx + 1] = u[i - 1, j]
        if ty == TILE_DIM - 1 or i == i0 + ni - 1:
            tile[ty + 2, tx + 1] = u[i + 1, j]
        if tx == 0:
            tile[ty + 1, 0] = u[i, j - 1]
        if tx == TILE_DIM - 1 or j == j0 + nj - 1:
            tile[ty + 1, tx + 2] = u[i, j + 1]
    cuda.syncthreads()

    if active:
        c = tile[ty + 1, tx + 1]
        n = tile[ty, tx + 1]
        s = tile[ty + 2, tx + 1]
        w = tile[ty + 1, tx]
        e = tile[ty + 1, tx + 2]
        kc = kappa[i, j]
        kn = 0.5 * (kc + kappa[i - 1, j])
        ks = 0.5 * (kc + kappa[i + 1, j])
        kw = 0.5 * (kc + kappa[i, j - 1])
        ke = 0.5 * (kc + kappa[i, j + 1])
        div = (ke * (e - c) - kw * (c - w) + ks * (s - c) - kn * (c - n)) * inv_dx2
        out[i, j] = c + dt * (div + S[i, j]) - dt * damp * c


REGION_FULL = (1, 1, LOCAL_N, LOCAL_N)
REGION_DEEP = (2, 2, LOCAL_N - 2, LOCAL_N - 2)            # [2..255]^2
REGIONS_BORDER = (
    (1, 1, 1, LOCAL_N),                                    # top row
    (LOCAL_N, 1, 1, LOCAL_N),                              # bottom row
    (2, 1, LOCAL_N - 2, 1),                                # left column
    (2, LOCAL_N, LOCAL_N - 2, 1),                          # right column
)


def make_gpu_state(ctx: dict, f: dict) -> dict:
    return dict(
        d_u=cuda.to_device(f["u0"]),
        d_out=cuda.to_device(f["u0"].copy()),
        d_kappa=cuda.to_device(f["kappa"]),
        d_S=cuda.to_device(f["S"]),
        d_send=cuda.device_array(4 * LOCAL_N, dtype=np.float64),
        d_recv=cuda.to_device(np.zeros(4 * LOCAL_N)),
        send_h=cuda.pinned_array(4 * LOCAL_N, dtype=np.float64),
        recv_h=cuda.pinned_array(4 * LOCAL_N, dtype=np.float64),
        stream_interior=cuda.stream(),
        stream_boundary=cuda.stream(),
    )


def launch_stencil(gs, f, region, stream=0):
    i0, j0, ni, nj = region
    grid = ((nj + TILE_DIM - 1) // TILE_DIM, (ni + TILE_DIM - 1) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        gs["d_u"], gs["d_kappa"], gs["d_S"], gs["d_out"],
        f["dt"], f["inv_dx2"], LAMBDA_DAMP, i0, j0, ni, nj,
    )


def post_irecv(ctx, gs):
    c, r, n = ctx["cart_comm"], gs["recv_h"], LOCAL_N
    return [
        c.Irecv(r[0:n], source=ctx["up"], tag=0),
        c.Irecv(r[n:2 * n], source=ctx["down"], tag=1),
        c.Irecv(r[2 * n:3 * n], source=ctx["left"], tag=2),
        c.Irecv(r[3 * n:4 * n], source=ctx["right"], tag=3),
    ]


def post_isend(ctx, gs):
    c, s, n = ctx["cart_comm"], gs["send_h"], LOCAL_N
    return [
        c.Isend(s[0:n], dest=ctx["down"], tag=0),
        c.Isend(s[n:2 * n], dest=ctx["up"], tag=1),
        c.Isend(s[2 * n:3 * n], dest=ctx["right"], tag=2),
        c.Isend(s[3 * n:4 * n], dest=ctx["left"], tag=3),
    ]


PACK_GRID = (LOCAL_N + 127) // 128


def exchange_blocking(ctx, gs):
    pack_halos_kernel[PACK_GRID, 128](gs["d_u"], gs["d_send"])
    gs["d_send"].copy_to_host(gs["send_h"])
    reqs = post_irecv(ctx, gs) + post_isend(ctx, gs)
    MPI.Request.Waitall(reqs)
    gs["d_recv"].copy_to_device(gs["recv_h"])
    unpack_halos_kernel[PACK_GRID, 128](gs["d_u"], gs["d_recv"])
    cuda.synchronize()


def step_serialized(ctx, f, gs):
    exchange_blocking(ctx, gs)
    launch_stencil(gs, f, REGION_FULL)
    cuda.synchronize()
    gs["d_u"], gs["d_out"] = gs["d_out"], gs["d_u"]


def step_overlapped(ctx, f, gs):
    si, sb = gs["stream_interior"], gs["stream_boundary"]
    recv_reqs = post_irecv(ctx, gs)
    pack_halos_kernel[PACK_GRID, 128, sb](gs["d_u"], gs["d_send"])
    gs["d_send"].copy_to_host(gs["send_h"], stream=sb)
    launch_stencil(gs, f, REGION_DEEP, stream=si)          # overlaps with D2H + MPI
    sb.synchronize()
    send_reqs = post_isend(ctx, gs)
    MPI.Request.Waitall(recv_reqs + send_reqs)
    gs["d_recv"].copy_to_device(gs["recv_h"], stream=sb)
    unpack_halos_kernel[PACK_GRID, 128, sb](gs["d_u"], gs["d_recv"])
    for region in REGIONS_BORDER:
        launch_stencil(gs, f, region, stream=sb)
    sb.synchronize()
    si.synchronize()
    gs["d_u"], gs["d_out"] = gs["d_out"], gs["d_u"]


def global_energy(ctx, f, u_host):
    e_loc = float(np.sum(u_host[1:-1, 1:-1] ** 2)) * f["dx"] * f["dx"]
    return ctx["cart_comm"].allreduce(e_loc, op=MPI.SUM)


def run_mode(ctx, f, mode, steps, energy_at=()):
    step = step_overlapped if mode == "overlapped" else step_serialized
    gs = make_gpu_state(ctx, f)
    energies = {}
    if 0 in energy_at:
        energies[0] = global_energy(ctx, f, f["u0"])
    ctx["comm"].Barrier()
    cuda.synchronize()
    elapsed = 0.0
    for s in range(1, steps + 1):
        t0 = time.perf_counter()
        step(ctx, f, gs)
        elapsed += time.perf_counter() - t0
        if s in energy_at:
            energies[s] = global_energy(ctx, f, gs["d_u"].copy_to_host())
    ctx["comm"].Barrier()
    ms = ctx["comm"].allreduce(elapsed * 1e3, op=MPI.MAX)
    return dict(u=gs["d_u"].copy_to_host(), ms=ms, energies=energies)


def verify_halo_exchange(ctx, f):
    """Ghost edges received from neighbours must equal the analytic initial field."""
    u_z = f["u0"].copy()
    u_z[0, :] = 0.0
    u_z[-1, :] = 0.0
    u_z[:, 0] = 0.0
    u_z[:, -1] = 0.0
    gs = make_gpu_state(ctx, f)
    gs["d_u"].copy_to_device(u_z)
    exchange_blocking(ctx, gs)
    got = gs["d_u"].copy_to_host()
    err = max(
        np.abs(got[0, 1:-1] - f["u0"][0, 1:-1]).max(),
        np.abs(got[-1, 1:-1] - f["u0"][-1, 1:-1]).max(),
        np.abs(got[1:-1, 0] - f["u0"][1:-1, 0]).max(),
        np.abs(got[1:-1, -1] - f["u0"][1:-1, -1]).max(),
    )
    return ctx["cart_comm"].allreduce(float(err), op=MPI.MAX)


# ==============================================================================
# TASK 4: host false-sharing kernel (no register hoisting) + stride sweep
# ==============================================================================
@intrinsic
def _volatile_inc(typingctx, arr, idx):
    """shm[idx] += 1 as monotonic-atomic load + store so LLVM cannot keep it in a register."""
    if not (isinstance(arr, types.Array) and isinstance(idx, types.Integer)):
        return None
    sig = types.uint64(arr, idx)

    def codegen(context, builder, signature, args):
        arrty, idxty = signature.args
        ary = context.make_array(arrty)(context, builder, args[0])
        i = context.cast(builder, args[1], idxty, types.intp)
        ptr = cgutils.get_item_pointer(context, builder, arrty, ary, [i])
        old = builder.load_atomic(ptr, "monotonic", 8)
        new = builder.add(old, ir.Constant(old.type, 1))
        builder.store_atomic(new, ptr, "monotonic", 8)
        return new

    return sig, codegen


@njit(nogil=True)
def host_shm_accumulate(shm, idx, iters):
    for _ in range(iters):
        _volatile_inc(shm, idx)
    return shm[idx]


def _shm_sync(ctx):
    ctx["win"].Sync()
    ctx["node_comm"].Barrier()
    ctx["win"].Sync()


def benchmark_shm_strides(ctx: dict) -> dict:
    node, shm, nr = ctx["node_comm"], ctx["shm_view"], ctx["node_rank"]
    host_shm_accumulate(shm, nr, 1000)               # JIT warm-up
    _shm_sync(ctx)
    results = {}
    for stride in (1, 8, 16):
        lats = []
        for _ in range(SHM_REPS):
            if nr == 0:
                shm[:] = 0
            _shm_sync(ctx)
            t0 = time.perf_counter()
            host_shm_accumulate(shm, nr * stride, SHM_ITERS)
            t1 = time.perf_counter()
            _shm_sync(ctx)
            lats.append(node.allreduce((t1 - t0) * 1e3, op=MPI.MAX))
            if int(shm[nr * stride]) != SHM_ITERS:
                raise RuntimeError(f"Lost updates at stride={stride} on rank {nr}.")
        results[stride] = float(np.median(lats))
    return results


# ==============================================================================
# Audits
# ==============================================================================
def audit_ptx() -> dict:
    asm = tiled_stencil_kernel.inspect_asm()
    text = next(iter(asm.values())) if isinstance(asm, dict) else asm
    return {
        "st.shared.f64": len(re.findall(r"st\.shared\.f64", text)),
        "ld.shared.f64": len(re.findall(r"ld\.shared\.f64", text)),
        "bar.sync": len(re.findall(r"bar\.sync", text)),
    }


def audit_llvm() -> dict:
    ir_txt = host_shm_accumulate.inspect_llvm()
    text = next(iter(ir_txt.values())) if isinstance(ir_txt, dict) else ir_txt
    return {
        "load_atomic": len(re.findall(r"load atomic", text)),
        "store_atomic": len(re.findall(r"store atomic", text)),
        "add_i64": len(re.findall(r"\badd\b.*\bi64\b", text)),
    }


# ==============================================================================
# TASK 5: main
# ==============================================================================
def main():
    if len(sys.argv) < 2:
        raise SystemExit("usage: hybrid_mpi_cuda_cache.py <github_username>")
    ctx = init_hybrid_context(sys.argv[1])
    cart = ctx["cart_comm"]
    root = ctx["rank"] == 0
    say = (lambda *a: print(*a, flush=True)) if root else (lambda *a: None)

    f = initialize_fields(ctx)
    say(f"[ctx] user={ctx['github_user']} gpu={ctx['gpu_uuid']} cpu={ctx['cpu_sig']} "
        f"line={ctx['cpu_line_bytes']}B seed={ctx['seed']}")
    say(f"[cfl] kappa_max={f['kappa_max']:.6f} dx={f['dx']:.9e} dt_max={f['dt_max']:.6e} "
        f"dt={f['dt']:.6e} courant_ok={f['dt'] <= f['dt_max']}")
    if f["dt"] > f["dt_max"]:
        raise RuntimeError("CFL violated")

    halo_err = verify_halo_exchange(ctx, f)
    say(f"[halo] max ghost error vs analytic field = {halo_err:.3e}")
    if halo_err != 0.0:
        raise RuntimeError("Halo exchange orientation/tag mismatch")

    run_mode(ctx, f, "serialized", 3)                # warm-up / JIT
    run_mode(ctx, f, "overlapped", 3)
    energy_steps = (0, 100, 200, 300)
    ser = run_mode(ctx, f, "serialized", STENCIL_STEPS, energy_steps)
    ovl = run_mode(ctx, f, "overlapped", STENCIL_STEPS, energy_steps)

    local_diff = float(np.max(np.abs(ser["u"][1:-1, 1:-1] - ovl["u"][1:-1, 1:-1])))
    max_diff = cart.allreduce(local_diff, op=MPI.MAX)
    say(f"[parity] max_diff = {max_diff:.3e}")
    if not max_diff < 1e-12:
        raise RuntimeError("Overlapped and serialized fields diverge")

    say(f"[timing] gpu_serialized_ms={ser['ms']:.3f} gpu_overlapped_ms={ovl['ms']:.3f} "
        f"speedup={ser['ms'] / ovl['ms']:.3f}")
    for s in energy_steps:
        say(f"[energy] step {s:3d}: serialized={ser['energies'][s]:.12e} "
            f"overlapped={ovl['energies'][s]:.12e}")
    ratios = [ser["energies"][s] for s in energy_steps]
    say(f"[energy] non-increasing={all(b <= a * (1 + 1e-9) for a, b in zip(ratios, ratios[1:]))}")

    shm = benchmark_shm_strides(ctx)
    say(f"[shm] stride=1 (8B)={shm[1]:.3f} ms | stride=8 (64B)={shm[8]:.3f} ms | "
        f"stride=16 (128B)={shm[16]:.3f} ms")
    say(f"[shm] false_sharing_slowdown(1/8)={shm[1] / shm[8]:.3f}x  "
        f"padding_ratio(8/16)={shm[8] / shm[16]:.3f}x")

    ptx = audit_ptx() if root else None
    llvm = audit_llvm()
    llvm_all = ctx["comm"].gather(llvm, root=0)
    if root:
        say(f"[ptx] {ptx}")
        say(f"[llvm] {llvm}")
        if ptx["st.shared.f64"] < 1 or ptx["ld.shared.f64"] < 1 or ptx["bar.sync"] < 1:
            raise RuntimeError("PTX audit failed")
        if llvm["load_atomic"] < 1 or llvm["store_atomic"] < 1:
            raise RuntimeError("LLVM audit failed: update was hoisted into a register")

        manifest = dict(
            github_username=ctx["github_user"], gpu_uuid=ctx["gpu_uuid"],
            cpu_signature=ctx["cpu_sig"], cpu_line_bytes=ctx["cpu_line_bytes"],
            instance_seed=ctx["seed"], mpi_ranks=4, global_grid=[GLOBAL_NX, GLOBAL_NY],
            steps=STENCIL_STEPS, kappa_max=f["kappa_max"], dx=f["dx"],
            dt=f["dt"], dt_max=f["dt_max"], courant_ok=bool(f["dt"] <= f["dt_max"]),
            max_diff=max_diff, halo_error=halo_err,
            gpu_serialized_ms=ser["ms"], gpu_overlapped_ms=ovl["ms"],
            energy_serialized={str(k): v for k, v in ser["energies"].items()},
            energy_overlapped={str(k): v for k, v in ovl["energies"].items()},
            shm_latency_ms={"stride_1": shm[1], "stride_8": shm[8], "stride_16": shm[16]},
            false_sharing_slowdown=shm[1] / shm[8], padding_ratio_8_over_16=shm[8] / shm[16],
            ptx_audit=ptx, llvm_audit=llvm, llvm_audit_all_ranks=llvm_all,
        )
        body = json.dumps(manifest, sort_keys=True, indent=2)
        manifest["manifest_sha256"] = hashlib.sha256(body.encode("utf-8")).hexdigest()
        with open("hybrid_lab_manifest.json", "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, sort_keys=True, indent=2)
        say("[manifest] hybrid_lab_manifest.json written")

    ctx["win"].Unlock_all()
    ctx["win"].Free()


if __name__ == "__main__":
    main()


In [ ]:
import subprocess
cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "hybrid_mpi_cuda_cache.py", GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")

In [ ]:
!cat hybrid_lab_manifest.json

### Analytical & Forensic Engineering Responses

> Замените все ⟦…⟧ числами из вашего вывода ячейки запуска / manifest.

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):** The row pitch is 258 × 8 = 2,064 B, so consecutive elements of `u[1:257, 1]` are 2,064 B apart. On the host a 64-B line holds 8 doubles, but a column walk touches a new line per element: 256 lines × 64 B = 16,384 B moved for 2,048 B useful (8× amplification). On the T4 each 8-B load in a strided warp lands in its own 32-B L2 sector: 256 × 32 B = 8,192 B for 2,048 B useful (4× amplification, 6,144 B wasted per column, 12,288 B for both left/right columns per step). Without `pack_halos_kernel` the column would have to leave the device as 256 separate 8-B DMA pieces (or a `cudaMemcpy2D` with 8-B width and 2,064-B pitch). The kernel does the strided gather once inside device DRAM and the PCIe transfer becomes a single contiguous 2,048-B copy per halo (4 × 2,048 = 8,192 B total per step), so the wasted transfer overhead (≈ 6,144 B/column of DRAM sector traffic plus 256 DMA descriptors) is avoided on the PCIe side.

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):** From `tiled_stencil_kernel.inspect_asm()`: `st.shared.f64` = ⟦ptx_st⟧, `ld.shared.f64` = ⟦ptx_ld⟧, `bar.sync` = ⟦ptx_bar⟧. A 16 × 16 block has 256 threads = 8 warps (warp w holds rows ty = 2w, 2w+1). The tile is written cooperatively: every thread stores its centre cell and edge threads store halo cells. A thread at row ty reads `tile[ty+2, tx+1]`, which was stored by the thread at row ty+1, often in the next warp (e.g. ty = 1 → warp 0 reads what warp 1 writes). Warps are scheduled independently, so without `cuda.syncthreads()` this is a Read-After-Write hazard: the reading warp can load stale or uninitialised shared memory before the writing warp has executed its `st.shared`, giving nondeterministic stencil values (also affects the halo cells at tile edges).

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):** Measured `gpu_serialized_ms` = ⟦ser_ms⟧ ms, `gpu_overlapped_ms` = ⟦ovl_ms⟧ ms (speed-up ⟦speedup⟧×, for 300 steps, max over ranks). Four MPI processes on one T4 each own a separate CUDA context; without MPS the contexts are time-sliced, so kernels from different ranks do not truly run concurrently, and all four ranks' D2H/H2D copies share the same PCIe link and the same few DMA copy engines, so the "overlapped" copies queue behind each other. The halo path is also staged: device → pinned host → MPI shared-memory copy → pinned host → device. With four dedicated GPUs each rank has its own PCIe link/copy engines and GPUDirect RDMA/P2P lets the NIC or peer GPU DMA straight from device memory, removing host staging and the host-side serialisation, so the overlap hides communication much better.

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):** Measured: stride = 1 (8 B) ⟦t1⟧ ms, stride = 8 (64 B) ⟦t8⟧ ms, stride = 16 (128 B) ⟦t16⟧ ms; false-sharing slowdown = t1 / t8 = ⟦ratio⟧×. With stride = 1, `shm_view[0]` and `shm_view[1]` are in the same 64-B line (aligned window). Trace: initially the line is I (or E in one core). Rank 0's core A writes `shm_view[0]`: it issues an RFO, other copies → I, A holds the line in **M**. Rank 1's core B writes `shm_view[1]`: B's RFO snoops A; A's M line is forwarded/written back and goes **M → I**, B gets it **I → M**. Rank 0's next write misses again: B **M → I**, A **I → M**. Every update therefore pays a cache-to-cache transfer even though the two ranks touch different words; if a core only reads, the holder goes M → S and the next write needs another invalidation S → I. The LLVM audit (⟦llvm_load⟧ `load atomic`, ⟦llvm_store⟧ `store atomic`) confirms every update really hits memory, i.e. nothing was hoisted into a register.

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):** Measured stride = 8 (64 B) = ⟦t8⟧ ms vs stride = 16 (128 B) = ⟦t16⟧ ms (ratio ⟦ratio816⟧×). The adjacent-cache-line (spatial) prefetcher treats memory as 128-B aligned pairs of 64-B lines (even/odd) and tends to fetch the buddy line together with the demanded one. At stride = 8 rank k uses line 2k′ and its buddy may be pulled in/invalidated by a neighbour's pair, so some residual coherence traffic can remain; at stride = 16 each rank owns a whole 128-B sector, which removes that effect. The cost of over-padding is footprint: each useful 8-B counter occupies 128 B (16× expansion vs 8 B), so with many counters fewer useful words fit in L1/L2 and capacity/conflict misses grow, and the extra prefetched buddy line is wasted bandwidth. For only four counters the footprint is negligible, so the extra gain is small.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):** Δx = 1/513 = 1.9493 × 10⁻³, Δx² = 3.7998 × 10⁻⁶. With κ_max = ⟦kappa_max⟧: Δt_max = Δx² / (4 κ_max) = ⟦dt_max⟧. The simulation uses Δt = 0.8 · Δt_max = ⟦dt⟧ < Δt_max, so the explicit scheme is stable (Courant condition satisfied). Global energy E = Σu²Δx² from `cart_comm.allreduce`: step 0 = ⟦E0⟧, step 100 = ⟦E100⟧, step 200 = ⟦E200⟧, step 300 = ⟦E300⟧ (serialized and overlapped runs agree). The decrease reflects diffusion through the Dirichlet boundary plus the λ = 0.15 damping term (the small localized source adds a negligible amount).